# n_bits vs ROC-AUC
Cross-validated ROC-AUC over fingerprint size.


In [1]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

MODEL_DIR = 'xgboost'
MODEL_NAME = 'XGBoost'

cwd = Path.cwd()
project_root = Path("..").resolve() if cwd.name == MODEL_DIR else cwd
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from qspr_config import (
    DATA_PATH,
    ECFP_RADIUS,
    N_BITS_GRID,
    N_ESTIMATORS,
    RANDOM_SEED,
    CV_FOLDS,
    N_JOBS,
    FIG_DPI,
    FIGSIZE_SQUARE,
)
from qspr_common import (
    load_dataset,
    build_feature_matrix,
    make_binary_target,
    apply_plot_style,
    resolve_output_dir,
)

apply_plot_style()
out_dir = resolve_output_dir(MODEL_DIR)


In [2]:
df_raw = load_dataset(DATA_PATH)


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

mean_scores = []
std_scores = []
for n_bits in N_BITS_GRID:
    df_bits, X = build_feature_matrix(df_raw, radius=ECFP_RADIUS, n_bits=n_bits)
    y, _ = make_binary_target(df_bits["Solubility"].to_numpy())

    model = XGBClassifier(
        n_estimators=N_ESTIMATORS,
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="auc",
        verbosity=0,
    )
    scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc", n_jobs=N_JOBS)
    mean_scores.append(scores.mean())
    std_scores.append(scores.std())

fig, ax = plt.subplots(figsize=FIGSIZE_SQUARE)
ax.errorbar(N_BITS_GRID, mean_scores, yerr=std_scores, marker="o", capsize=3)
ax.set_xlabel("n_bits")
ax.set_ylabel("ROC-AUC")
ax.set_title(f"{MODEL_NAME}: n_bits vs ROC-AUC")
ax.set_xticks(N_BITS_GRID)
ax.tick_params(axis="x", rotation=45)

out_path = out_dir / "n_bits_vs_roc_auc.png"
fig.tight_layout()
fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
plt.show()
out_path
